In [ ]:
# ============================================================
# STEP 7: LoRA fine-tuning (Colab, T4 GPU zaroori)
# ============================================================
EXP    = "B"
SMOKE  = False
RESUME = False

import os, sys, json
REPO_URL = "https://github.com/FatimaaShaheen/urdu-asr-subtitles.git"
REPO_DIR = "/content/urdu-asr-subtitles"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# versions pin kiye hain taake code in se match kare
!pip install -q "transformers>=4.46,<5" "peft>=0.13" "accelerate>=1.0" "datasets<4" jiwer librosa soundfile pyyaml
!pip install -q -U "torchao>=0.16.0"

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

import torch, yaml, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from datasets import load_from_disk, concatenate_datasets
from google.colab import drive, files
assert torch.cuda.is_available(), "GPU on karo: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/urdu_asr"

from src.training.lora_whisper import run_experiment
cfg = yaml.safe_load(open(f"{REPO_DIR}/configs/train_config.yaml"))

# ---------- Data ----------
train = load_from_disk(f"{DRIVE}/merged_train")
if EXP == "A":          # A = sirf FLEURS, B = merged (FLEURS + UrduSpeech)
    train = train.filter(lambda s: s == "fleurs", input_columns=["source"])
train = train.filter(lambda d: d <= 30.0, input_columns=["duration"])   # Whisper 30s limit

fleurs_val = load_from_disk(f"{DRIVE}/fleurs_val")
urdu_val = load_from_disk(f"{DRIVE}/urduspeech_val")
val = concatenate_datasets([
    fleurs_val.select(range(min(60, len(fleurs_val)))),
    urdu_val.select(range(min(40, len(urdu_val)))),
])   # monitoring ke liye chhota mixed validation, A aur B dono ke liye ek jaisa

if SMOKE:
    train = train.shuffle(seed=0).select(range(64)); val = val.select(range(16))
    cfg["train"].update(max_steps=10, eval_steps=5, save_steps=5,
                        logging_steps=1, warmup_steps=2)

hours = sum(train["duration"]) / 3600
print(f"\nExperiment {EXP}: {len(train)} train examples ({hours:.2f} ghante), "
      f"{len(val)} val | sources: {dict(Counter(train['source']))}")
print("Audio decode check:", train[0]["audio"]["array"].shape)

# ---------- Training ----------
OUT = f"{DRIVE}/runs/{EXP}{'_smoke' if SMOKE else ''}"
log, info = run_experiment(EXP, train, val, OUT, cfg, resume=RESUME)
info.update(hours=round(hours, 2), smoke=SMOKE, config=cfg)
print("\n", json.dumps({k: v for k, v in info.items() if k != "config"}, indent=1))

# ---------- Natije mehfooz ----------
RES = f"/content/results_step7_{EXP}"
os.makedirs(RES, exist_ok=True)
df = pd.DataFrame(log)
df.to_csv(f"{RES}/train_log_{EXP}.csv", index=False, encoding="utf-8-sig")
json.dump(info, open(f"{RES}/run_info_{EXP}.json", "w"), indent=1)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
if "loss" in df:
    tr = df[df["loss"].notna()]
    ax[0].plot(tr["step"], tr["loss"]); ax[0].set_title(f"{EXP}: training loss"); ax[0].set_xlabel("step")
if "eval_wer" in df:
    ev = df[df["eval_wer"].notna()]
    ax[1].plot(ev["step"], ev["eval_wer"], marker="o"); ax[1].set_title(f"{EXP}: validation WER"); ax[1].set_xlabel("step")
plt.tight_layout(); plt.savefig(f"{RES}/train_curves_{EXP}.png", dpi=120); plt.show()

if "eval_wer" in df:
    print(df[df["eval_wer"].notna()][["step", "eval_loss", "eval_wer", "eval_cer"]])

if not SMOKE:
    !cd /content && zip -qr results_step7_{EXP}.zip results_step7_{EXP}
    !mkdir -p {DRIVE}/results && cp /content/results_step7_{EXP}.zip {DRIVE}/results/
    files.download(f"/content/results_step7_{EXP}.zip")
print("\nAdapter yahan save hua:", f"{OUT}/adapter")